In [1]:
"""
alameda_parcels.py  --  parcel ownership / land-use / water-use classification for the Alameda SLR study.

Axes (kept separate so precedence never silently decides an answer):
    owner_form         WHO holds title      public / association / nonprofit_religious / business / utility / trust / estate / individual
    tenure_structure   HOW title is held    fee_simple / condo_unit / common_area / coop / pud_fee_simple
    ownership_structure  headline label for maps (your original vocabulary; see ownership_structure())
    land_use           WHAT the parcel is for -- STRICTLY from Assessor use codes (no name-based guessing)
    water_use_class    relationship to the water by USE/HOLDING (code, name evidence, or curated override) -- NOT SLR exposure
    owner_locality     where the tax bill goes: on-site / Alameda / elsewhere in CA / out of state

Design rules
  * explicit column names, fail loudly if missing; blank strings are treated as missing
  * filters are FLAGS, not deletions; every classification records the rule that fired (`*_basis`)
  * name-based evidence never changes `land_use`; it is recorded in water_use_* with a confidence level
  * the core (`classify_parcels`) needs only pandas/numpy so it can be unit-tested without GDAL;
    geopandas / matplotlib helpers are imported lazily.
"""
from __future__ import annotations

import re
from pathlib import Path

import numpy as np
import pandas as pd

# --------------------------------------------------------------------------------------
# Configuration
# --------------------------------------------------------------------------------------
GDB_LAYER = "PARCEL_With_Ownership"
EXPECTED_CRS_EPSG = 2227                   # NAD83 / California zone 3 (US survey ft)
ALAMEDA_TRA = {"21"}                       # traprimary for the City of Alameda (== situscity 'ALAMEDA' in the 2026-09-15 extract)
ALAMEDA_APN_BOOKS = {"69", "70", "71", "72", "73", "74"}
LARGE_PARCEL_SQFT = 1_000_000

REQUIRED_COLUMNS = [
    "apn", "book", "traprimary", "usecode", "ownersname", "totalnetvalue", "hoex",
    "situsstreetnumber", "situsstreetname", "mailingaddressstreet", "mailingaddresscity", "mailingaddressstate",
]
USECODE_CSV = Path(__file__).with_name("usecodes.csv")   # parsed from the Assessor's Use Code PDF


# --------------------------------------------------------------------------------------
# Small helpers
# --------------------------------------------------------------------------------------
def _has(s: pd.Series, pattern: str) -> pd.Series:
    """Boolean Series: does `pattern` match anywhere in each string? (compiled once; no pandas capture-group warning)"""
    rx = re.compile(pattern)
    return s.map(lambda x: bool(rx.search(x)))


def blank_to_na(df: pd.DataFrame) -> pd.DataFrame:
    """This extract stores many missing values as '' -- convert blank/whitespace strings to NA in text columns."""
    out = df.copy()
    for c in out.columns:
        if pd.api.types.is_object_dtype(out[c]) or pd.api.types.is_string_dtype(out[c]):
            s = out[c].astype("string")
            out[c] = s.where(s.str.strip().ne(""), pd.NA)
    return out


# --------------------------------------------------------------------------------------
# Use-code lookup (single source of truth = the Assessor's table, not hand-typed lists)
# --------------------------------------------------------------------------------------
def load_usecodes(path: Path = USECODE_CSV) -> pd.DataFrame:
    lk = pd.read_csv(path, dtype=str)
    lk["usecode"] = lk["usecode"].str.zfill(4)
    lk["code_int"] = lk["usecode"].astype(int)
    d = lk["use_desc"].str.lower()
    # '... Tract with Common Area' (1800, 1850) describes the HOME parcel; only 'common area [or use]' as its own
    # designation (1166, 1190, 1590, 3990, 7390 ...) is a parcel owned in common.
    lk["is_common_area"] = d.str.contains(r"(?<!with )(?<!w/)common area", regex=True)
    lk["is_condo"] = d.str.contains(r"condominium", regex=True) & ~lk["is_common_area"]
    lk["is_coop"] = d.str.contains(r"cooperative", regex=True) & ~lk["is_common_area"]
    lk["is_pud"] = d.str.contains(r"planned development", regex=True) & ~lk["is_common_area"]
    lk["is_vacant"] = d.str.contains(r"\bvacant\b", regex=True) | (lk["code_int"] == 840)   # 0840 = tract land (vacant)
    return lk.set_index("usecode")


def tenure_structure(lk: pd.DataFrame) -> pd.Series:
    """How title is held, derived from the use-code description."""
    t = pd.Series("fee_simple", index=lk.index)
    t[lk.is_pud] = "pud_fee_simple"          # individually owned lot/home, HOA-governed common elements
    t[lk.is_coop] = "coop"
    t[lk.is_condo] = "condo_unit"
    t[lk.is_common_area] = "common_area"     # parcel owned in common (HOA / association)
    return t


# Land-use buckets. Order matters (first match wins). Sets are explicit so every choice is reviewable.
_UTILITY = {400, 500}
_GOVERNMENT = {300, 6001, 6100}
_MARITIME = {9901}
_RECREATION = {6300, 9600, 9700, 9800, 9900, 9905, 9910}
_LODGING = {8900, 9000}
_PARKING_AUTO = {8000, 8100, 8200, 8300, 8400, 8500, 4900}
_MIXED_USE = {1950, 3200, 3705, 4240, 7301, 7302, 7321, 7322, 7341, 7342, 7391, 7392, 7705, 7706}
_OFFICE_RD = {4201, 4202, 4205, 9300, 9400, 9401, 9405, 9491, 9500}
_CIVIC_EXTRA = {8700, 8800, 8801, 8802}
_RES_EXTRA = {600, 700, 750, 900, 940, 5100, 5200, 8901, 9100}
RESIDENTIAL_LAND_USES = ("Residential", "Residential w/ Commercial Mixed-Use")


def land_use_from_code(uc: int, is_vacant: bool) -> str:
    if uc in _UTILITY:      return "Public Utility & Infrastructure"
    if uc in _GOVERNMENT:   return "Government (exempt public agency)"
    if uc in _MARITIME:     return "Maritime & Submerged Lands"
    if is_vacant:           return "Vacant Land"
    if uc in _RECREATION:   return "Recreation & Entertainment"
    if uc in _MIXED_USE:    return "Residential w/ Commercial Mixed-Use"
    if uc in _OFFICE_RD:    return "Office & R&D"
    if uc == 6850:          return "Service & Retail/Exchange"        # 'historical commercial'
    if 6000 <= uc < 7000 or uc in _CIVIC_EXTRA: return "Civic & Institutional"
    if uc in _LODGING:      return "Lodging"
    if uc in _PARKING_AUTO: return "Parking & Auto Services"
    if 4000 <= uc < 5000 or uc == 9902: return "Industrial & Logistics"
    if 5300 <= uc < 5900:   return "Rural / Agricultural"
    if 1100 <= uc < 3000 or 7000 <= uc < 8000 or uc in _RES_EXTRA: return "Residential"
    if 3000 <= uc < 4000 or 8000 <= uc < 10000: return "Service & Retail/Exchange"
    return "Other / Unclassified"


def residential_type(uc: int) -> str | None:
    if uc == 750: return "floating_home"
    if uc in (1100, 1101, 1120, 1130, 1140, 1150, 1200, 1201, 1300, 1900, 1901): return "single_family"
    if 7300 <= uc < 7400: return "condo"
    if uc in (7400, 7430): return "coop"
    if 1400 <= uc < 1700 or 1800 <= uc < 1900: return "townhouse_pud_duet"
    if 2100 <= uc < 3000: return "2_to_4_units"
    if uc in (7100, 7200, 7500, 7700, 7701, 7705, 7706, 7790, 7800): return "5plus_units"
    if uc in (600, 700, 900, 940, 1700, 5100, 5200, 7600, 7900, 8901, 9100): return "other_residential"
    return None


# --------------------------------------------------------------------------------------
# Owner classification
# --------------------------------------------------------------------------------------
def normalize_name(s: pd.Series) -> pd.Series:
    """Upper-case, strip punctuation, collapse whitespace. Keeps '&' and digits."""
    s = s.fillna("").astype(str).str.upper()
    s = s.str.replace(r"[.,'\"]", "", regex=True)
    s = s.str.replace(r"\bCO[- ]?(TR|TRS|TRUSTEES?|TTEES?)\b", "TRS", regex=True)   # 'CO TR' = co-trustee, not 'company'
    return s.str.replace(r"\s+", " ", regex=True).str.strip()


# (level, agency label, regex) -- top to bottom, first match wins. Anchored to the START of the name unless noted, so
# 'CHEVRON U S A INC', 'HOME DEPOT USA INC' or a surname containing 'BART' cannot match.
# `public_level` keeps city / county / special_district apart for drill-down; the headline label merges them
# into 'Public Entities - Local/Regional'.
PUBLIC_RULES: list[tuple[str, str, str]] = [
    ("federal", "US Navy / Dept of Defense", r"^(US NAVY|DEPT OF (THE )?(NAVY|ARMY|AIR FORCE)|DEPARTMENT OF (THE )?(NAVY|ARMY|AIR FORCE|DEFENSE))\b"),
    ("federal", "US Coast Guard", r"^(UNITED STATES )?COAST GUARD\b"),
    ("federal", "US Postal Service", r"^(UNITED STATES POSTAL|US POSTAL|USPS)\b"),
    ("federal", "United States of America", r"^(UNITED STATES(?! (STEEL|CELLULAR|FIRE|GYPSUM|SHIPPING))|US GOVT|US GOVERNMENT|U S GOVT|U S A( ETAL)?$|USA( ETAL)?$|GENERAL SERVICES ADMIN)"),
    ("special_district", "BART", r"RAPID TRANSIT DISTRICT|^BART\b"),
    ("special_district", "AC Transit", r"^(ALAMEDA[- ]?CONTRA COSTA TRANSIT|A ?C TRANSIT DISTRICT)(?! EMPLOYEES)"),
    ("special_district", "EBMUD", r"^(EAST BAY MUNICIPAL UTILITY|EBMUD)"),
    ("special_district", "East Bay Regional Park District", r"^EAST BAY REGIONAL PARK"),
    ("special_district", "Alameda Co. Flood Control District", r"^ALAMEDA (COUNTY|CO) FLOOD"),
    ("special_district", "School district", r"\b(UNIFIED|SCHOOL) DISTRICT\b"),
    ("special_district", "Peralta Community College District", r"^PERALTA (JUNIOR |COMMUNITY )+COLLEGE"),
    ("special_district", "Housing authority", r"HOUSING AUTHORITY|^(CITY )?(OF )?(ALAMEDA|OAKLAND) HOUSING( ETAL)?$|^CITY (OF )?(OAKLAND|ALAMEDA) HOUSING"),
    ("special_district", "Health care district", r"HEALTH CARE DISTRICT"),
    ("special_district", "Port of Oakland", r"^PORT OF OAKLAND"),
    ("special_district", "Joint powers / energy authority", r"^(EAST BAY COMMUNITY ENERGY|NORTHERN CALIFORNIA POWER) AUTHORITY|^NORTHERN CALIFORNIA POWER AGENCY"),
    ("state", "State of California", r"^(STATE OF CALIF|STATE CALIF|CALIFORNIA STATE OF\b|CALIFORNIA STATE LANDS|STATE LANDS COMMISSION|CALTRANS|PEOPLE OF THE STATE)"),
    ("state", "University of California", r"^(REGENTS.*UNIVERSITY.*CALIF|UNIVERSITY OF CALIF)"),
    ("county", "County of Alameda", r"^(COUNTY OF ALAMEDA|ALAMEDA COUNTY(?! (COMMUNITY FOOD|WATER|TRANSPORTATION)))\b"),
    ("city", "City of Alameda", r"^(CITY OF ALAMEDA|CITY ALAMEDA|ALAMEDA CITY OF|COMMUNITY IMPROVEMENT COMMISSION)\b"),
    ("city", "City of Oakland", r"^(CITY OF OAKLAND|CITY OAKLAND|OAKLAND CITY OF|OAKLAND REDEVELOPMENT)\b"),
    ("city", "City of San Leandro / Hayward", r"^CITY (OF )?(SAN LEANDRO|HAYWARD)\b"),
]

LOAN_ASSN_RE = r"\b(SAVINGS (&|AND) LOAN|LOAN ASSOCIATION|NATIONAL ASSOCIATION|MORTGAGE ASSOCIATION)\b"
GENERIC_ASSN_RE = r"\b(ASSN|ASSOC|ASSOCIATION)$"          # any remaining name that ENDS in 'association'
ASSOCIATION_RE = (r"\b(HOMEOWNERS?|HOME OWNERS?|OWNERS?|CONDOMINIUMS?|CONDO|TOWNHOUSES?|TOWNHOMES?|COMMUNITY|MAINTENANCE)\b"
                  r".*\b(ASSN|ASSO|ASSOC|ASSOCIATION)\b|\bHOA\b|^COMMON AREA")
NONPROFIT_RE = (r"\b(CHURCH|BAPTIST|CATHOLIC|DIOCESE|SYNAGOGUE|TEMPLE|MOSQUE|CONGREGATION|MINISTR(Y|IES)|LUTHERAN|"
                r"PRESBYTERIAN|METHODIST|EPISCOPAL|BUDDHIST|ASSEMBLY OF GOD|JEHOVAH|SALVATION ARMY|YMCA|YWCA|"
                r"FOUNDATION|CHARIT(Y|IES)|GIRLS INC|FOOD BANK|MUSEUM|HABITAT FOR HUMANITY|FAMILY SERVICES|HOSPICE|"
                r"AFFORDABLE HOUSING (CORP|CORPORATION)|ALLIANCE FOR HEALTH|FIRST 5|SCHOOL|ACADEMY|"
                r"CLUB|SOCIETY|GUILD|CHAPEL|CHABAD|RELIGION|COLLABORATIVE)\b")
BUSINESS_RE = (r"\bL L C$|\b(LLC|LLP|LP|LTD|INC|INCORPORATED|CORP|CORPORATION|COMPANY|CO|PARTNERS|PARTNERSHIP|ASSOCIATES|"
               r"HOLDINGS?|PROPERTIES|PROPERTY|INVESTMENTS?|INVESTORS|VENTURES?|ENTERPRISES?|REALTY|DEVELOPMENTS?|"
               r"DEVELOPERS|CAPITAL|GROUP|FUND|BANK|MORTGAGE|FINANCIAL|EQUITIES|INDUSTRIES|LEASING|MANAGEMENT|"
               r"BUILDERS|APARTMENTS|HOMES)\b")
TRUST_RE = r"\b(TRUST|TRUSTS|TR|TRS|TRUSTEE|TRUSTEES|TTEE|TTEES|TSTEE|TSTEES|TRST)\b"
ESTATE_RE = r"\b(ESTATE OF|EST OF|HEIRS OF)\b|\bESTATE$"


def classify_owner(names: pd.Series, usecodes: pd.Series) -> pd.DataFrame:
    """Return owner_form, public_level, public_agency, owner_form_basis for each row."""
    n = normalize_name(names)
    idx = n.index
    form = pd.Series("", index=idx, dtype=object)
    basis = pd.Series("", index=idx, dtype=object)
    level = pd.Series(pd.NA, index=idx, dtype=object)
    agency = pd.Series(pd.NA, index=idx, dtype=object)

    def assign(mask, f, b):
        m = mask & (form == "")
        form[m] = f
        basis[m] = b
        return m

    assign(n == "", "unknown", "blank owner name")
    for lvl, ag, pat in PUBLIC_RULES:                                   # 1. public, by name
        m = assign(_has(n, pat), "public", f"name:{ag}")
        level[m] = lvl
        agency[m] = ag
    assign(_has(n, LOAN_ASSN_RE), "business", "name:bank / savings & loan")   # 2. legal-form rules, in precedence order
    assign(_has(n, ASSOCIATION_RE) | _has(n, GENERIC_ASSN_RE), "association", "name:HOA/association")
    assign(_has(n, NONPROFIT_RE), "nonprofit_religious", "name:nonprofit/religious")
    assign(_has(n, BUSINESS_RE), "business", "name:business suffix/keyword")
    assign(_has(n, TRUST_RE), "trust", "name:trust marker")
    assign(_has(n, ESTATE_RE), "estate", "name:estate")
    uc = usecodes.fillna("")                                            # 3. use-code fallbacks (only if no name rule fired)
    m = assign(uc.isin(["0300", "6001", "6100"]), "public", "usecode:exempt/government, agency unresolved")
    level[m] = "unresolved"
    assign(uc.isin(["0400", "0500"]), "utility", "usecode:utility")
    assign(pd.Series(True, index=idx), "individual", "default: no organisational marker")   # 4. default
    return pd.DataFrame({"owner_form": form, "public_level": level, "public_agency": agency,
                         "owner_form_basis": basis, "owner_name_norm": n})


# Headline label: YOUR original vocabulary, now fed by the corrected logic.
PUBLIC_LABEL = {"federal": "Public Entities - Federal", "state": "Public Entities - State",
                "county": "Public Entities - Local/Regional", "city": "Public Entities - Local/Regional",
                "special_district": "Public Entities - Local/Regional", "unresolved": "Public Entities - Unresolved"}


def ownership_structure(owner_form: pd.Series, public_level: pd.Series, tenure: pd.Series) -> pd.Series:
    s = pd.Series("Individual Ownership", index=owner_form.index, dtype=object)
    s[owner_form == "unknown"] = "Unknown / No Assessor Record"
    s[owner_form == "nonprofit_religious"] = "Nonprofit & Religious Entities"
    s[owner_form.isin(["business", "utility"])] = "Corporate Entities"
    s[owner_form == "trust"] = "Trust Entities"
    shared = tenure.isin(["condo_unit", "common_area", "coop"]) | (owner_form == "association")
    s[shared & (owner_form != "public")] = "Shared Ownership"        # condo/HOA/co-op beats legal form of the unit owner
    pub = owner_form == "public"
    s[pub] = public_level[pub].map(PUBLIC_LABEL)
    return s


# --------------------------------------------------------------------------------------
# Jurisdiction + flags
# --------------------------------------------------------------------------------------
def jurisdiction(df: pd.DataFrame) -> pd.Series:
    tra = df["traprimary"].astype("string").str.strip()
    book = df["book"].astype("string").str.strip()
    apn = df["apn"].astype("string")
    is_ph = apn.str.startswith("999-", na=False)
    j = pd.Series("Other jurisdiction", index=df.index, dtype=object)
    j[book.isin(ALAMEDA_APN_BOOKS) & tra.isna()] = "Alameda APN book, no assessor record"
    j[book.isin(ALAMEDA_APN_BOOKS) & tra.notna() & ~tra.isin(ALAMEDA_TRA)] = "Alameda APN book, other TRA (review)"
    j[is_ph] = "Placeholder APN (book 999)"
    j[tra.isin(ALAMEDA_TRA)] = "Alameda"
    return j


def add_flags(df: pd.DataFrame, area_sqft: pd.Series) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)
    apn = df["apn"].astype("string")
    out["area_sqft"] = area_sqft
    out["is_placeholder_apn"] = apn.str.endswith("-999-99", na=False)
    out["no_assessor_record"] = df["ownersname"].isna() & df["usecode"].isna()
    out["is_large_exempt"] = (area_sqft > LARGE_PARCEL_SQFT) & (df["totalnetvalue"].fillna(0) == 0)
    # 'submerged_candidate' is a *candidate* flag only -- confirm with a shoreline / land mask or your exposure layers.
    out["submerged_candidate"] = out["is_placeholder_apn"] | (out["no_assessor_record"] & (area_sqft > 100_000))
    if {"centroid_x", "centroid_y"}.issubset(df.columns):     # condo stacks share a centroid
        out["stack_size"] = df.groupby(["centroid_x", "centroid_y"])["apn"].transform("size").fillna(1).astype("int32")
    out["apn_polygon_count"] = df.groupby("apn")["apn"].transform("size").fillna(1).astype("int32")
    return out


# --------------------------------------------------------------------------------------
# Owner occupancy and owner locality
# --------------------------------------------------------------------------------------
_STREET_SUFFIX = {"AVENUE": "AVE", "AV": "AVE", "STREET": "ST", "BOULEVARD": "BLVD", "ROAD": "RD", "DRIVE": "DR",
                  "LANE": "LN", "COURT": "CT", "PLACE": "PL", "PARKWAY": "PKWY", "CIRCLE": "CIR", "TERRACE": "TER",
                  "HIGHWAY": "HWY", "SQUARE": "SQ", "NORTH": "N", "SOUTH": "S", "EAST": "E", "WEST": "W"}


def normalize_street(s: pd.Series) -> pd.Series:
    """'875 ISLAND DRIVE, STE A' -> '875 ISLAND DR' (suffix abbreviations unified, unit designators dropped)."""
    s = s.fillna("").astype(str).str.upper()
    s = s.str.replace(r"[.,#']", "", regex=True).str.replace(r"\s+", " ", regex=True).str.strip()
    s = s.str.replace(r"\s(STE|SUITE|UNIT|APT|APARTMENT|SPC|SPACE|BLDG|FL|FLOOR)\s.*$", "", regex=True)
    return s.map(lambda x: " ".join(_STREET_SUFFIX.get(t, t) for t in x.split(" ")))


def add_occupancy_and_locality(df: pd.DataFrame, residential_mask: pd.Series) -> pd.DataFrame:
    """
    owner_occupied_hoex : homeowners' exemption claimed (the legally-grounded owner-occupancy signal). Residential only.
    owner_on_site       : the tax-bill (mailing) street address equals the property's street address.
                          Residences -> owner-occupied.  Commercial -> owner likely receives mail at the property
                          (owner-user or on-site office).  Does NOT prove tenancy either way; corporate owners can
                          use a registered agent or manager's office.
    owner_locality      : On-site / Alameda, off-site / Elsewhere in California / Out of state / Unknown  (ALL parcel types)
    is_local_owner      : on-site or mailing address in Alameda, CA
    registration_type   : your original Local/External label (mailing city == Alameda, CA)
    """
    out = pd.DataFrame(index=df.index)
    hoex = df["hoex"].fillna(0)
    oo = pd.Series(pd.NA, index=df.index, dtype="boolean")
    oo[residential_mask] = (hoex > 0)[residential_mask]
    out["owner_occupied_hoex"] = oo

    sit = normalize_street(df["situsstreetnumber"].astype("string").fillna("") + " " + df["situsstreetname"].fillna(""))
    mail = normalize_street(df["mailingaddressstreet"])
    on_site = ((sit == mail) & sit.str.match(r"^\d")).astype(bool)
    city = df["mailingaddresscity"].astype("string").str.upper().str.strip()
    state = df["mailingaddressstate"].astype("string").str.upper().str.strip()
    b = lambda x: x.fillna(False).to_numpy(dtype=bool)
    in_alameda = b((city == "ALAMEDA") & (state == "CA"))
    out["owner_on_site"] = on_site
    out["owner_locality"] = np.select(
        [b(city.isna() | state.isna()) & ~on_site.to_numpy(), on_site.to_numpy(), in_alameda, b(state == "CA")],
        ["Unknown", "On-site", "Alameda, off-site", "Elsewhere in California"],
        default="Out of state / non-US")
    out["is_local_owner"] = on_site | pd.Series(in_alameda, index=df.index)
    out["registration_type"] = np.select([b(city.isna() | state.isna()), in_alameda],
                                         ["Unknown", "Locally Registered"], default="Externally Registered")
    return out


# --------------------------------------------------------------------------------------
# Water use (by USE / HOLDING; not SLR exposure)
# --------------------------------------------------------------------------------------
WATER_CLASSES = ["floating_home", "boat_berth", "salt_pond", "yacht_club", "boatyard_shipyard", "marina_harbor",
                 "submerged_candidate", "other_water_related"]
_CODE_WATER = {"0750": ("floating_home", "Assessor use code 0750 (floating home)"),
               "9901": ("boat_berth", "Assessor use code 9901 (privately owned boat berth)"),
               "4700": ("salt_pond", "Assessor use code 4700 (salt ponds)")}
# Name evidence applies ONLY to entity owners (business / nonprofit-club), fee-simple, non-residential parcels --
# never to individuals or trusts (first names like MARINA and BERTHA, surnames like BERTH...).
_WATER_NAME_RULES = [
    ("yacht_club", "owner name: boating club", r"\b(YACHT|SAILING|BOAT|ROWING|CANOE) CLUB\b"),
    ("boatyard_shipyard", "owner name: boatyard / shipyard", r"\b(BOATYARD|BOAT YARD|BOATWORKS|BOAT WORKS|SHIPYARD|SHIP YARD|DRYDOCK|DRY DOCK)\b"),
    ("marina_harbor", "owner name: marina / harbor", r"\bMARINA\b(?! VILLAGE)|\bYACHT HARBOR\b|\bBOAT HARBOR\b|\bFORTMAN BASIN\b"),
]
_WATER_ENTITY_FORMS = {"business", "nonprofit_religious"}
# Weak signals -> REVIEW LIST only (never auto-classified). 'HARBOR BAY' and 'MARINA VILLAGE' are business-park names.
_WATER_WEAK_TERMS = r"\b(HARBOR|HARBOUR|BASIN|LAGOON|WHARF|PIER|DOCKS?|MARINE|MARITIME|WATERFRONT|ESTUARY|SHORELINE|SEAPLANE|FERRY|TERMINALS?)\b"


def classify_water_use(out: pd.DataFrame, overrides: pd.DataFrame | None = None) -> pd.DataFrame:
    """Adds water_use_class / _confidence / _basis, hosts_floating_homes, water_dependent, water_use_review."""
    idx = out.index
    cls = pd.Series(pd.NA, index=idx, dtype=object)
    conf = pd.Series(pd.NA, index=idx, dtype=object)
    basis = pd.Series(pd.NA, index=idx, dtype=object)

    def setw(mask, c, cf, b):
        m = mask & cls.isna()
        cls[m], conf[m], basis[m] = c, cf, b

    for code, (c, b) in _CODE_WATER.items():                                   # 1. Assessor use code (zero-padded strings)
        setw(out["usecode"] == code, c, "code", b)
    entity = out["owner_form"].isin(_WATER_ENTITY_FORMS)
    eligible = entity & (out["tenure_structure"] == "fee_simple") & ~out["land_use"].isin(RESIDENTIAL_LAND_USES)
    n = out["owner_name_norm"]
    for c, b, pat in _WATER_NAME_RULES:                                        # 2. entity-name evidence (inferred)
        setw(eligible & _has(n, pat), c, "name", b)
    setw(out["submerged_candidate"], "submerged_candidate", "geometry",       # 3. placeholder / unassessed polygons
         "placeholder or unassessed large polygon")

    hosts = pd.Series(False, index=idx)
    src = pd.Series(pd.NA, index=idx, dtype=object)
    lu_over = pd.Series(pd.NA, index=idx, dtype=object)
    if overrides is not None and len(overrides):                               # 4. curated overrides win
        ov = overrides.copy()
        ov.columns = [c.strip().lower() for c in ov.columns]
        if "confirmed_class" in ov.columns:
            # The edited review file works as-is. It carries BOTH the automatic label (`water_use_class`) and the analyst's
            # `confirmed_class`; only the analyst's column is an override, so the automatic label is discarded here.
            ov = ov.drop(columns=[c for c in ("water_use_class",) if c in ov.columns]).rename(
                columns={"confirmed_class": "water_use_class"})
        need = {"apn", "water_use_class"}
        if not need.issubset(ov.columns):
            raise KeyError(f"overrides need columns {sorted(need)}; got {list(ov.columns)}")
        ov["water_use_class"] = ov["water_use_class"].astype("string").str.strip().str.lower()
        ov = ov[ov["water_use_class"].notna() & ov["water_use_class"].ne("")]      # rows you have not reviewed are ignored
        bad = sorted(set(ov["water_use_class"]) - set(WATER_CLASSES) - {"none"})
        if bad:
            raise ValueError(f"unknown water_use_class in overrides: {bad}; allowed: {WATER_CLASSES + ['none']}")
        ov = ov.drop_duplicates("apn", keep="last").set_index("apn")
        m = out["apn"].isin(ov.index)
        ocls = out.loc[m, "apn"].map(ov["water_use_class"])
        cls[m] = ocls.where(ocls != "none", pd.NA)                            # 'none' = analyst rejects an auto flag
        conf[m] = np.where(ocls == "none", pd.NA, "override")
        basis[m] = np.where(ocls == "none", pd.NA, "curated override")
        if "hosts_floating_homes" in ov.columns:
            hosts[m] = out.loc[m, "apn"].map(ov["hosts_floating_homes"]).map(
                lambda v: str(v).strip().lower() in ("1", "true", "yes", "y", "t")).astype(bool)
        if "source" in ov.columns:
            src[m] = out.loc[m, "apn"].map(ov["source"])
        if "land_use_override" in ov.columns:
            lu_over[m] = out.loc[m, "apn"].map(ov["land_use_override"])

    weak = (entity & _has(n, _WATER_WEAK_TERMS) & out["land_use"].isin(
        ["Vacant Land", "Industrial & Logistics", "Recreation & Entertainment", "Civic & Institutional",
         "Parking & Auto Services"])) | (entity & (out["usecode"] == "9900")) | out["is_large_exempt"]
    res = pd.DataFrame({
        "water_use_class": cls, "water_use_confidence": conf, "water_use_basis": basis,
        "hosts_floating_homes": hosts, "water_use_source": src, "land_use_override": lu_over})
    res["water_dependent"] = res["water_use_class"].notna()
    res["water_use_review"] = weak & ~res["water_dependent"] & (conf != "override")
    return res


def water_use_review_table(out: pd.DataFrame) -> pd.DataFrame:
    """Rows an analyst should look at: every auto-classified water-use parcel (to confirm/reject) plus weak candidates.
    Fill `confirmed_class` (one of WATER_CLASSES, or 'none' to reject an auto flag), optionally `hosts_floating_homes`
    (yes/no) and `source`. Pass the edited file straight to classify_parcels(water_overrides=...) -- rows with a blank
    `confirmed_class` are ignored, and an override always beats the automatic rules."""
    m = out["include_default"] & (out["water_dependent"] | out["water_use_review"])
    cols = ["apn", "situsaddress", "ownersname", "usecode", "use_desc", "land_use", "ownership_structure", "area_sqft",
            "water_use_class", "water_use_confidence", "water_use_basis", "is_large_exempt", "units"]
    t = out.loc[m, [c for c in cols if c in out.columns]].copy()
    t["acres"] = (t.pop("area_sqft") / 43560).round(2)
    for c in ("confirmed_class", "hosts_floating_homes", "source", "note"):
        t[c] = ""
    return t.sort_values(["water_use_confidence", "ownersname", "apn"], na_position="last").reset_index(drop=True)


# --------------------------------------------------------------------------------------
# Main entry: pure pandas
# --------------------------------------------------------------------------------------
def classify_parcels(df: pd.DataFrame, area_sqft: pd.Series | None = None, lookup: pd.DataFrame | None = None,
                     water_overrides: pd.DataFrame | str | Path | None = None) -> pd.DataFrame:
    """Add all derived columns. `df` needs REQUIRED_COLUMNS. Returns a new DataFrame (input untouched)."""
    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        raise KeyError(f"Required columns missing: {missing}")
    lk = lookup if lookup is not None else load_usecodes()
    out = blank_to_na(df)
    if area_sqft is None:
        area_sqft = out["shape_Area"] if "shape_Area" in out.columns else out.geometry.area
    out["usecode"] = out["usecode"].astype("string").str.strip().str.zfill(4).where(out["usecode"].notna())

    out["jurisdiction"] = jurisdiction(out)
    out["include_default"] = out["jurisdiction"].isin(
        ["Alameda", "Alameda APN book, other TRA (review)", "Alameda APN book, no assessor record",
         "Placeholder APN (book 999)"])
    flags = add_flags(out, area_sqft)
    out = out.join(flags[[c for c in flags.columns if c not in out.columns]])

    unknown_codes = sorted(set(out["usecode"].dropna()) - set(lk.index))
    if unknown_codes:
        raise ValueError(f"Use codes not in the Assessor table: {unknown_codes}")
    out["use_desc"] = out["usecode"].map(lk["use_desc"])
    out["tenure_structure"] = out["usecode"].map(tenure_structure(lk)).fillna("unknown")
    out["hoa_governed"] = out["tenure_structure"].isin(["condo_unit", "common_area", "coop", "pud_fee_simple"])
    code_int = out["usecode"].map(lk["code_int"])
    vac = out["usecode"].map(lk["is_vacant"]).fillna(False).astype(bool)
    out["land_use"] = [land_use_from_code(int(c), v) if pd.notna(c) else "No use code (unassessed / placeholder)"
                       for c, v in zip(code_int, vac)]
    out.loc[out["submerged_candidate"] & out["usecode"].isna(), "land_use"] = "Maritime & Submerged Lands"
    out["residential_type"] = [residential_type(int(c)) if pd.notna(c) else None for c in code_int]

    out = out.join(classify_owner(out["ownersname"], out["usecode"]))
    out["ownership_structure"] = ownership_structure(out["owner_form"], out["public_level"], out["tenure_structure"])

    res_mask = out["land_use"].isin(RESIDENTIAL_LAND_USES)
    out = out.join(add_occupancy_and_locality(out, res_mask))

    if isinstance(water_overrides, (str, Path)):
        p = Path(water_overrides)
        water_overrides = pd.read_csv(p, dtype=str) if p.exists() else None
    out = out.join(classify_water_use(out, water_overrides))
    has_lu = out["land_use_override"].notna()
    out.loc[has_lu, "land_use"] = out.loc[has_lu, "land_use_override"]       # only ever set by a curated override
    out = out.drop(columns="land_use_override")
    return out


# --------------------------------------------------------------------------------------
# Validation and notes
# --------------------------------------------------------------------------------------
def validate(out: pd.DataFrame, expect_nonempty=("Public Entities - Federal", "Public Entities - State",
                                                  "Public Entities - Local/Regional", "Shared Ownership",
                                                  "Corporate Entities", "Trust Entities")) -> list[str]:
    """Hard problems only (empty buckets usually mean a rule/column bug). Empty list == OK."""
    problems = []
    inc = out[out["include_default"]]
    counts = inc["ownership_structure"].value_counts()
    for b in expect_nonempty:
        if counts.get(b, 0) == 0:
            problems.append(f"ownership_structure '{b}' is EMPTY -- rule or column bug?")
    if inc["land_use"].eq("Other / Unclassified").any():
        problems.append("some use codes fell into 'Other / Unclassified'")
    if inc["owner_form_basis"].eq("").any():
        problems.append("some rows received no owner_form")
    return problems


def data_notes(out: pd.DataFrame) -> list[str]:
    """Informational findings that are properties of THIS extract, not bugs."""
    inc = out[out["include_default"]]
    notes = []
    if not (inc["usecode"] == "0750").any():
        notes.append("No use-code 0750 (floating home) parcels in this extract. Floating homes / liveaboards cannot be "
                     "identified from the parcel file alone -- supply curated overrides (see water_use_review_table).")
    if not (inc["usecode"] == "9901").any():
        notes.append("No use-code 9901 (privately owned boat berth) parcels inside Alameda in this extract.")
    n_name = int((inc["water_use_confidence"] == "name").sum())
    notes.append(f"{n_name} parcels are water-related by owner-name evidence only (inferred; confirm before relying on them).")
    v = inc[(inc["land_use"] == "Vacant Land") & inc["water_dependent"]]
    if len(v):
        notes.append(f"{len(v)} parcels coded as 'Vacant Land' are flagged water_dependent (boatyard/marina land the "
                     "Assessor codes as vacant industrial) -- exclude them if 'Vacant Land' means blank slate.")
    return notes


# --------------------------------------------------------------------------------------
# geopandas / matplotlib helpers (imported lazily; NOT exercised by the unit tests)
# --------------------------------------------------------------------------------------
OWNERSHIP_COLORS = {   # draw order = dict order; later entries are painted on top
    "Unknown / No Assessor Record": "#d9d9d9", "Individual Ownership": "#8da0cb", "Trust Entities": "#66c2a5",
    "Corporate Entities": "#fc8d62", "Nonprofit & Religious Entities": "#e78ac3",
    "Public Entities - Local/Regional": "#a6d854", "Public Entities - State": "#ffd92f",
    "Public Entities - Federal": "#e5c494", "Public Entities - Unresolved": "#b3b3b3", "Shared Ownership": "#1b7837"}
WATER_COLORS = {"floating_home": "#08306b", "boat_berth": "#2171b5", "marina_harbor": "#4292c6", "boatyard_shipyard": "#6baed6",
                "yacht_club": "#9ecae1", "salt_pond": "#c6dbef", "submerged_candidate": "#bdbdbd", "other_water_related": "#41ab5d"}


def read_parcels(gdb_path):
    import geopandas as gpd
    gdf = gpd.read_file(Path(gdb_path), layer=GDB_LAYER)
    if gdf.crs is None or gdf.crs.to_epsg() != EXPECTED_CRS_EPSG:
        raise ValueError(f"Unexpected CRS {gdf.crs}; expected EPSG:{EXPECTED_CRS_EPSG}")
    return gdf


def build_master(gdb_path, water_overrides=None):
    import geopandas as gpd
    gdf = read_parcels(gdb_path)
    out = classify_parcels(pd.DataFrame(gdf.drop(columns="geometry")), area_sqft=gdf.geometry.area,
                           water_overrides=water_overrides)
    out = gpd.GeoDataFrame(out, geometry=gdf.geometry.values, crs=gdf.crs)
    problems = validate(out)
    if problems:
        raise AssertionError("; ".join(problems))
    return out


def map_layer(gdf, priority_col="ownership_structure", priority_value="Shared Ownership"):
    """De-duplicate stacked footprints (keep the priority class) and return rows in DRAW order.
    matplotlib paints rows in order, so priority rows must come LAST to end up on top."""
    key = gdf.geometry.to_wkb()
    pri = (gdf[priority_col] == priority_value).astype(int)
    d = gdf.assign(_pri=pri, _key=key).sort_values("_pri", ascending=False).drop_duplicates("_key")
    return d.sort_values("_pri", ascending=True).drop(columns=["_pri", "_key"])


def plot_categorical(ax, gdf, column, colors, title, base=None, base_color="#eeeeee"):
    """Stable, explicit colours per category (same category = same colour on every map). `base` is an optional
    GeoDataFrame drawn first in a neutral colour (e.g. all parcels under a water-use map)."""
    import matplotlib.patches as mpatches
    if base is not None:
        base.plot(ax=ax, color=base_color, edgecolor="none")
    handles = []
    for cat, col in colors.items():
        sub = gdf[gdf[column] == cat]
        if len(sub):
            sub.plot(ax=ax, color=col, edgecolor="black", linewidth=0.05)
            handles.append(mpatches.Patch(color=col, label=f"{cat} ({len(sub):,})"))
    unexpected = sorted(set(gdf[column].dropna()) - set(colors))
    if unexpected:
        print(f"[plot_categorical] categories with no assigned colour, not drawn: {unexpected}")
    ax.legend(handles=handles, loc="upper left", fontsize=8, title=column)
    ax.set_title(title, fontsize=16)
    ax.set_axis_off()

NameError: name '__file__' is not defined